# 02. Tiền xử lý Dữ liệu, Kỹ thuật Đặc trưng (FE) & Chia tập
**Đề tài:** Dự đoán khả năng khách hàng hủy đặt phòng khách sạn (*Hotel Booking Demand*)  
**Giả định thời điểm dự đoán:** Tại thời điểm xác nhận đặt phòng (*Booking Confirmation*)

## Mục tiêu của Notebook
1. **Đồng bộ Data Cleaning với `01_eda.ipynb`:** Loại các bản ghi bất hợp lệ xác định trước (`confirmed_zero_guests`, `adr < 0`), giữ `adr = 5400`, `total_nights == 0`, giữ `children = NA` và giữ các bản ghi trùng lặp (không có `booking_id`).
2. **Kiểm soát Data Leakage:** Loại bỏ *Direct Leakage* (`reservation_status`, `reservation_status_date`) và *Temporal / Post-booking Leakage* (`assigned_room_type`, `booking_changes`, `days_in_waiting_list`).
3. **Xử lý Missing cố định & Feature Engineering tinh gọn:** Chỉ áp dụng các quy tắc cố định không học tham số từ phân phối; mọi phép imputation học tham số (`SimpleImputer`) và mã hóa (`OneHotEncoder`, `TargetEncoder`, `StandardScaler`) được fit bên trong từng Training Fold ở Notebook 03.
4. **Chia tập kép bảo toàn tính tổng quát:** Giữ đồng thời **Stratified Random Split (Group-aware 80/20)** chống trùng fingerprint giữa Train/Test và **Out-of-Time (OOT) Split** theo thời gian (`05/2017`).

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

RANDOM_STATE = 42
TARGET = "is_canceled"
RAW_PATH = Path("../data/raw/hotel_bookings.csv")
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

df_raw = pd.read_csv(RAW_PATH)
print(f"Dữ liệu gốc (read-only): {df_raw.shape[0]:,} dòng x {df_raw.shape[1]} cột")
print(f"Tỷ lệ hủy ban đầu: {df_raw[TARGET].mean() * 100:.2f}%")

Dữ liệu gốc (read-only): 119,390 dòng x 32 cột
Tỷ lệ hủy ban đầu: 37.04%


## 1. Data Cleaning (Đồng bộ với Notebook 01)
Áp dụng đúng chính sách đã thống nhất ở `01_eda.ipynb` (tuyệt đối không dùng nhãn `is_canceled` để lọc dòng và không tự động xóa ngoại lai theo IQR):
- **Loại bỏ:** Bản ghi có tổng khách xác định bằng `0` (`adults + children + babies == 0` với `min_count=3`) và bản ghi có `adr < 0`.
- **Giữ lại:** Ngoại lai cực đại `adr = 5400`, bản ghi `0` đêm lưu trú (*day-use*), toàn bộ bản ghi trùng lặp (sẽ kiểm soát chồng lấn bằng fingerprint khi chia tập) và `4` giá trị thiếu của `children` (không mặc định `children = NA` là `0`).

In [2]:
df = df_raw.copy()
n_raw = len(df)

# Loại bỏ booking có tổng khách xác định bằng 0 (giữ nguyên các dòng có children=NA)
mask_zero_guests = df[["adults", "children", "babies"]].sum(axis=1, min_count=3).eq(0).fillna(False)
n_removed_guests = int(mask_zero_guests.sum())
df = df.loc[~mask_zero_guests].copy()

# Loại bỏ giá phòng ADR âm
mask_neg_adr = df["adr"].lt(0).fillna(False)
n_removed_neg_adr = int(mask_neg_adr.sum())
df = df.loc[~mask_neg_adr].reset_index(drop=True)

assert len(df) == n_raw - n_removed_guests - n_removed_neg_adr
assert df[TARGET].notna().all() and df[TARGET].isin([0, 1]).all()


clean_summary = pd.DataFrame([
    {"Bước": "Dữ liệu gốc (df_raw)", "Số dòng": n_raw, "Số dòng loại": 0, "Tỷ lệ hủy (%)": round(df_raw[TARGET].mean() * 100, 2)},
    {"Bước": "Sau loại confirmed_zero_guests & adr < 0", "Số dòng": len(df), "Số dòng loại": n_removed_guests + n_removed_neg_adr, "Tỷ lệ hủy (%)": round(df[TARGET].mean() * 100, 2)},
])
display(clean_summary)
print(f"Số giá trị thiếu ở cột children được giữ nguyên cho imputer trong CV: {int(df['children'].isna().sum())}")

,Bước,Số dòng,Số dòng loại,Tỷ lệ hủy (%)
0,Dữ liệu gốc (df_raw),119390,0,37.04
1,Sau loại confirmed_zero_guests & adr < 0,119209,181,37.08


Số giá trị thiếu ở cột children được giữ nguyên cho imputer trong CV: 4


## 2. Kiểm soát Data Leakage (Direct & Temporal Leakage)
Theo giả định dự đoán **tại thời điểm xác nhận đặt phòng**, ta rà soát trực tiếp các nhóm biến rủi ro trước khi tạo đặc trưng dẫn xuất (**lưu ý:** *row-level Feature Engineering không đồng nghĩa với Zero Leakage* — nếu biến nguồn chứa thông tin hậu đặt phòng thì biến dẫn xuất vẫn bị rò rỉ):

| Nhóm biến | Quyết định | Lý do |
|---|---|---|
| `reservation_status`, `reservation_status_date` | **Loại bỏ (Direct Leakage)** | Chỉ biết sau khi đơn hoàn tất hoặc bị hủy; khớp 100% với `is_canceled`. |
| `assigned_room_type`, `booking_changes`, `days_in_waiting_list` | **Loại bỏ (Temporal / Post-booking Leakage)** | Có thể thay đổi hoặc phát sinh sau mốc xác nhận đặt phòng; đồng thời không tạo biến dẫn xuất `room_type_changed`. |
| `deposit_type`, `adr`, `country`, `total_of_special_requests`, `required_car_parking_spaces` | **Giữ lại có lưu ý** | Chưa có bằng chứng khẳng định chắc chắn gây leakage nên giữ trong tập đặc trưng học thuật, kèm ghi chú cần xác minh mốc ghi nhận với hệ thống nguồn nếu triển khai thực tế. Không tạo thêm biến dẫn xuất khuếch đại từ `adr` (bỏ `adr_per_person`, `estimated_total_revenue`). |

In [3]:
direct_leakage_cols = ["reservation_status", "reservation_status_date"]
temporal_leakage_cols = ["assigned_room_type", "booking_changes", "days_in_waiting_list"]
dropped_leakage_cols = direct_leakage_cols + temporal_leakage_cols

df = df.drop(columns=dropped_leakage_cols)
assert not any(c in df.columns for c in dropped_leakage_cols), "Vẫn còn cột leakage trong dữ liệu!"

print(f"Đã loại {len(dropped_leakage_cols)} cột leakage: {dropped_leakage_cols}")
print(f"Kích thước sau khi loại leakage: {df.shape[0]:,} dòng x {df.shape[1]} cột")

Đã loại 5 cột leakage: ['reservation_status', 'reservation_status_date', 'assigned_room_type', 'booking_changes', 'days_in_waiting_list']
Kích thước sau khi loại leakage: 119,209 dòng x 27 cột


## 3. Xử lý Missing Cố định & Kỹ thuật Tạo Đặc trưng (Feature Engineering)
- **Xử lý thiếu cố định (không học tham số từ phân phối dữ liệu):**
  - `country`: điền nhãn phân loại riêng `'Unknown'`.
  - `agent`, `company`: tạo 2 cờ nhị phân `has_agent`, `has_company`, chuẩn hóa mã ID sang chuỗi và điền `'Unknown'` (không suy diễn mặc định là khách đặt trực tiếp).
  - `meal`: giữ nguyên giá trị `'Undefined'` như dữ liệu gốc.
  - `children`: **giữ nguyên `NaN`** để `SimpleImputer(strategy='median')` fit bên trong từng Training Fold ở Notebook 03.
- **Đặc trưng mới tinh gọn, có cơ sở nghiệp vụ:**
  - `total_nights`, `total_guests` (`min_count=3`, giữ `NaN` khi `children` thiếu), `weekend_ratio` (xử lý mẫu số `0` sạch bằng `np.where`).
  - `total_previous_bookings`, `prev_cancel_ratio` (xử lý mẫu số `0` sạch bằng `np.where`).
  - `arrival_month_num`, `arrival_month_sin`, `arrival_month_cos` (mã hóa chu kỳ tháng nhận phòng).

In [4]:
# 3.1 Xử lý thiếu cố định cho các biến phân loại / mã định danh
df["country"] = df["country"].fillna("Unknown").astype(str)

df["has_agent"] = df["agent"].notna().astype(int)
df["agent"] = df["agent"].apply(lambda x: str(int(x)) if pd.notna(x) else "Unknown")

df["has_company"] = df["company"].notna().astype(int)
df["company"] = df["company"].apply(lambda x: str(int(x)) if pd.notna(x) else "Unknown")

for col in ["hotel", "meal", "market_segment", "distribution_channel",
            "reserved_room_type", "deposit_type", "customer_type"]:
    df[col] = df[col].fillna("Unknown").astype(str)

# 3.2 Tạo các đặc trưng mới tinh gọn
month_map = {
    "January": 1, "February": 2, "March": 3, "April": 4,
    "May": 5, "June": 6, "July": 7, "August": 8,
    "September": 9, "October": 10, "November": 11, "December": 12,
}

df["total_nights"] = df["stays_in_weekend_nights"] + df["stays_in_week_nights"]
df["total_guests"] = df[["adults", "children", "babies"]].sum(axis=1, min_count=3)
df["weekend_ratio"] = np.where(
    df["total_nights"] > 0,
    df["stays_in_weekend_nights"] / df["total_nights"],
    0.0,
).round(4)

df["total_previous_bookings"] = df["previous_cancellations"] + df["previous_bookings_not_canceled"]
df["prev_cancel_ratio"] = np.where(
    df["total_previous_bookings"] > 0,
    df["previous_cancellations"] / df["total_previous_bookings"],
    0.0,
).round(4)

df["arrival_month_num"] = df["arrival_date_month"].map(month_map).astype(int)
df["arrival_month_sin"] = np.sin(2 * np.pi * df["arrival_month_num"] / 12).round(6)
df["arrival_month_cos"] = np.cos(2 * np.pi * df["arrival_month_num"] / 12).round(6)

# Tạo ngày nhận phòng dự kiến phục vụ chia OOT, sau đó loại cột text tháng thừa
arrival_date = pd.to_datetime(
    pd.DataFrame({
        "year": df["arrival_date_year"],
        "month": df["arrival_month_num"],
        "day": df["arrival_date_day_of_month"],
    }),
    errors="coerce",
)
assert arrival_date.notna().all(), "Phát hiện ngày nhận phòng không hợp lệ!"

df = df.drop(columns=["arrival_date_month"])

# Kiểm tra các cột còn giá trị thiếu (chỉ được phép nằm ở children và total_guests)
missing_after = df.isna().sum()
deferred_missing = missing_after[missing_after > 0]
print(f"Tổng số cột sau Feature Engineering (gồm target): {df.shape[1]}")
print("Các cột còn NaN trì hoãn cho imputer trong Notebook 03:")
display(deferred_missing.to_frame("missing_count"))

Tổng số cột sau Feature Engineering (gồm target): 36
Các cột còn NaN trì hoãn cho imputer trong Notebook 03:


,missing_count
children,4
total_guests,4


## 4. Chiến lược Chia tập Kép (Group-aware Stratified 80/20 & Out-of-Time)
1. **Stratified Random Split (80/20) chống trùng fingerprint:**
   - Vì ta giữ các dòng trùng lặp ở `df`, nếu chia ngẫu nhiên thông thường thì các mẫu có bộ đặc trưng giống hệt nhau sẽ xuất hiện đồng thời ở cả Train và Test.
   - Ta tính `fingerprint` dựa **chỉ trên các feature đầu vào thực tế** (không chứa `is_canceled` hay các cột leakage) và dùng `StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)` để tách `80/20` sao cho vừa cân bằng tỷ lệ nhãn, vừa đảm bảo **`0` fingerprint giao nhau** giữa `train_strat` và `test_strat`.
2. **Out-of-Time Split (OOT — `2017-05-01`):**
   - Chia chặt theo thứ tự thời gian của `arrival_date`: `train_oot` (`2015-07-01` đến `2017-04-30`) và `test_oot` (`2017-05-01` đến `2017-08-31`).
   - **Lưu ý quan trọng về mốc thời gian:** `arrival_date` là *ngày nhận phòng dự kiến*, khác với *ngày đặt phòng ước tính* (`arrival_date - lead_time`). Một số đơn nhận phòng sau `2017-05-01` có thể đã được đặt trước mốc này.
   - **Nguyên tắc đánh giá bắt buộc:** `test_oot` tuyệt đối **không** dùng cho training, tuning hay model selection; chỉ được đánh giá bằng mô hình huấn luyện riêng từ `train_oot` (không dùng mô hình fit trên `train_strat` vì `train_strat` chứa dữ liệu giai đoạn `05–08/2017`).

In [5]:
high_card_cat_cols = ["country", "agent", "company"]
low_card_cat_cols = [
    "hotel", "meal", "market_segment", "distribution_channel",
    "reserved_room_type", "deposit_type", "customer_type",
]
num_cols = [c for c in df.columns if c not in [TARGET] + high_card_cat_cols + low_card_cat_cols]
feature_cols = num_cols + low_card_cat_cols + high_card_cat_cols

# Tính fingerprint CHỈ trên các feature thực tế (không chứa target hay leakage)
fingerprint = pd.util.hash_pandas_object(df[feature_cols], index=False)

# 4.1 Stratified Random Split (Group-aware 80/20)
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
train_idx, test_idx = next(sgkf.split(df[feature_cols], df[TARGET], groups=fingerprint))
train_strat = df.iloc[train_idx].reset_index(drop=True)
test_strat = df.iloc[test_idx].reset_index(drop=True)

strat_fp_overlap = len(set(fingerprint.iloc[train_idx]).intersection(set(fingerprint.iloc[test_idx])))

# 4.2 Out-of-Time (OOT) Split theo arrival_date
oot_cutoff = pd.Timestamp("2017-05-01")
oot_train_mask = arrival_date < oot_cutoff
oot_test_mask = arrival_date >= oot_cutoff

train_oot = df.loc[oot_train_mask].reset_index(drop=True)
test_oot = df.loc[oot_test_mask].reset_index(drop=True)
oot_fp_overlap = len(set(fingerprint.loc[oot_train_mask]).intersection(set(fingerprint.loc[oot_test_mask])))

# Kiểm tra các assertions cơ bản
assert strat_fp_overlap == 0, f"Phát hiện {strat_fp_overlap} fingerprint chồng lấn ở Stratified Split!"
assert arrival_date.loc[oot_train_mask].max() < arrival_date.loc[oot_test_mask].min(), "OOT vi phạm thứ tự thời gian!"
unexpected_na = [c for c in df.columns if df[c].isna().any() and c not in ["children", "total_guests"]]
assert len(unexpected_na) == 0, f"Phát hiện NaN ngoài dự kiến ở cột: {unexpected_na}"

print(f"Phân nhóm đặc trưng: {len(num_cols)} số | {len(low_card_cat_cols)} low-card | {len(high_card_cat_cols)} high-card (tổng {len(feature_cols)} features)")
print(f"Fingerprint overlap — Stratified Split: {strat_fp_overlap} | Out-of-Time Split: {oot_fp_overlap}")
print(f"Mốc thời gian OOT: Train <= {arrival_date.loc[oot_train_mask].max().date()} | Test >= {arrival_date.loc[oot_test_mask].min().date()}")

Phân nhóm đặc trưng: 25 số | 7 low-card | 3 high-card (tổng 35 features)
Fingerprint overlap — Stratified Split: 0 | Out-of-Time Split: 0
Mốc thời gian OOT: Train <= 2017-04-30 | Test >= 2017-05-01


## 5. Xuất Dữ liệu Đã Xử lý & Cập nhật `feature_metadata.json`

In [6]:
train_strat.to_csv(PROCESSED_DIR / "train_strat.csv", index=False)
test_strat.to_csv(PROCESSED_DIR / "test_strat.csv", index=False)
train_oot.to_csv(PROCESSED_DIR / "train_oot.csv", index=False)
test_oot.to_csv(PROCESSED_DIR / "test_oot.csv", index=False)

metadata = {
    "random_state": RANDOM_STATE,
    "target_col": TARGET,
    "num_cols": num_cols,
    "low_card_cat_cols": low_card_cat_cols,
    "high_card_cat_cols": high_card_cat_cols,
    "dropped_leakage_cols": dropped_leakage_cols,
    "deferred_missing_cols": ["children", "total_guests"],
    "split_policy": {
        "stratified": "StratifiedGroupKFold(n_splits=5) grouped by feature fingerprint (0 overlap)",
        "oot_cutoff_date": str(oot_cutoff.date()),
        "oot_basis": "arrival_date",
    },
    "shapes": {
        "cleaned_total": list(df.shape),
        "train_strat": list(train_strat.shape),
        "test_strat": list(test_strat.shape),
        "train_oot": list(train_oot.shape),
        "test_oot": list(test_oot.shape),
    },
    "target_rates": {
        "cleaned_total": round(float(df[TARGET].mean()), 4),
        "train_strat": round(float(train_strat[TARGET].mean()), 4),
        "test_strat": round(float(test_strat[TARGET].mean()), 4),
        "train_oot": round(float(train_oot[TARGET].mean()), 4),
        "test_oot": round(float(test_oot[TARGET].mean()), 4),
    },
    "fingerprint_overlap": {
        "stratified_train_test": strat_fp_overlap,
        "oot_train_test": oot_fp_overlap,
    },
}
with (PROCESSED_DIR / "feature_metadata.json").open("w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

summary_splits = pd.DataFrame([
    {"Tập dữ liệu": "train_strat.csv", "Số dòng": len(train_strat), "Tỷ trọng (%)": round(len(train_strat)/len(df)*100, 2), "Số cột": train_strat.shape[1], "Tỷ lệ hủy (%)": round(train_strat[TARGET].mean()*100, 2)},
    {"Tập dữ liệu": "test_strat.csv", "Số dòng": len(test_strat), "Tỷ trọng (%)": round(len(test_strat)/len(df)*100, 2), "Số cột": test_strat.shape[1], "Tỷ lệ hủy (%)": round(test_strat[TARGET].mean()*100, 2)},
    {"Tập dữ liệu": "train_oot.csv (< 2017-05-01)", "Số dòng": len(train_oot), "Tỷ trọng (%)": round(len(train_oot)/len(df)*100, 2), "Số cột": train_oot.shape[1], "Tỷ lệ hủy (%)": round(train_oot[TARGET].mean()*100, 2)},
    {"Tập dữ liệu": "test_oot.csv (>= 2017-05-01)", "Số dòng": len(test_oot), "Tỷ trọng (%)": round(len(test_oot)/len(df)*100, 2), "Số cột": test_oot.shape[1], "Tỷ lệ hủy (%)": round(test_oot[TARGET].mean()*100, 2)},
])
display(summary_splits)
print("Đã xuất thành công 4 tệp CSV và feature_metadata.json vào data/processed/!")

,Tập dữ liệu,Số dòng,Tỷ trọng (%),Số cột,Tỷ lệ hủy (%)
0,train_strat.csv,95367,80.0,36,37.08
1,test_strat.csv,23842,20.0,36,37.08
2,train_oot.csv (< 2017-05-01),97032,81.4,36,36.28
3,test_oot.csv (>= 2017-05-01),22177,18.6,36,40.57


Đã xuất thành công 4 tệp CSV và feature_metadata.json vào data/processed/!


## 6. Ghi chú Cập nhật Cần thiết cho Notebook 03 & 04 (Đợt tiếp theo)
Vì dữ liệu đầu vào (`119.209` dòng giữ duplicates) và tập đặc trưng (`35` features sau khi loại nhóm biến hậu đặt phòng) đã thay đổi, toàn bộ kết quả mô hình cũ trong `models/` hiện không còn tương thích với dữ liệu mới và **cần được huấn luyện, đánh giá lại** ở đợt cập nhật Notebook 03 & 04:
1. **Notebook 03 (`03_modeling_tuning.ipynb`):**
   - Đổi chiến lược 5-Fold CV từ `StratifiedKFold` sang `StratifiedGroupKFold(n_splits=5)` nhóm theo `fingerprint` (tính trên `feature_cols` của `train_strat.csv`) để tránh trùng mẫu giữa train fold và validation fold.
   - Giữ `SimpleImputer(strategy='median')` trong `Pipeline` để xử lý tự động các giá trị `NaN` của `children` và `total_guests` ngay bên trong từng training fold.
   - Huấn luyện lại toàn bộ các mô hình cơ sở và tối ưu Optuna trên `train_strat.csv` mới.
2. **Notebook 04 (`04_evaluation_report.ipynb`):**
   - Chạy lại đánh giá toàn bộ mô hình mới trên `test_strat.csv` mới và cập nhật các biểu đồ đánh giá (`07`–`12`).
   - Giữ chặt quy tắc đánh giá Out-of-Time: chỉ huấn luyện lại kiến trúc mô hình tốt nhất trên `train_oot.csv` rồi mới đánh giá trên `test_oot.csv` (không dùng mô hình fit từ `train_strat.csv`).